# Notebook 01: Vector Addition — CUDA Basics

**Learning objectives**
- Understand the GPU thread hierarchy: Grid → Block → Thread
- Write your first `__global__` kernel function
- Use `cudaMalloc` / `cudaMemcpy` for host ↔ device data transfer
- Measure kernel performance in GB/s with CUDA Events
- Compare bandwidth utilization against PyTorch

**How to run:** Runtime > Change runtime type > **T4 GPU** → Runtime > **Run all**

*Reference diagrams: [docs/diagrams.md](../docs/diagrams.md) — Diagrams 2 (memory hierarchy), 3 (execution model), 5 (vector add sequence)*

In [ ]:
# Verify you have a T4 GPU attached
!nvidia-smi

## 1. The CUDA Thread Hierarchy

Before writing any code, build the mental model (Diagram 3 in docs/diagrams.md).

```
Grid  ── covers the whole problem  (N = 16,777,216 elements)
 ├── Block 0       threadIdx.x: 0…255,  blockIdx.x: 0
 ├── Block 1       threadIdx.x: 0…255,  blockIdx.x: 1
 ├── ...
 └── Block 65535   threadIdx.x: 0…255,  blockIdx.x: 65535
```

Each **thread** handles exactly one element: `C[i] = A[i] + B[i]`

The thread's global index is composed of two built-in variables:

```c
int i = blockIdx.x * blockDim.x + threadIdx.x;
//      └── which block?   └── block size   └── position within block
```

With `blockDim.x = 256` and `gridDim.x = 65536`:
- Total threads = 256 × 65536 = **16,777,216** — one per element
- Last block may have threads that exceed `N` → always guard with `if (i < N)`

> **Rule:** threads in the same *block* can share memory and synchronize.
> Threads in different blocks cannot communicate directly.

In [ ]:
%%writefile vec_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// ── Kernel: runs on GPU, launched from CPU ────────────────────────────────────
// __global__ qualifier means: called from host, executes on device
// Each of the N threads executes this function independently
__global__ void vecAdd(float *A, float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;  // global thread index
    if (i < N)                                        // boundary guard
        C[i] = A[i] + B[i];
}

// ── Host (CPU) code ───────────────────────────────────────────────────────────
int main() {
    const int    N     = 1 << 24;            // 16,777,216 elements
    const size_t bytes = N * sizeof(float);  // 64 MB per array

    // 1. Allocate and fill host memory
    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    // 2. Allocate device (GPU) memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // 3. Copy data: host -> device  (over PCIe, ~32 GB/s)
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // 4. Configure launch
    int blockSize = 256;                               // threads per block (multiple of 32)
    int gridSize  = (N + blockSize - 1) / blockSize;  // blocks per grid  (ceil division)

    // 5. Launch kernel  <<<grid, block>>>
    vecAdd<<<gridSize, blockSize>>>(d_A, d_B, d_C, N);

    // 6. Wait for GPU to finish (mandatory before D2H copy)
    cudaDeviceSynchronize();

    // 7. Copy result: device -> host
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    // 8. Verify all results
    int errors = 0;
    for (int i = 0; i < N; i++)
        if (fabsf(h_C[i] - 3.0f) > 1e-5f) errors++;

    printf("Grid:  %d blocks\n", gridSize);
    printf("Block: %d threads\n", blockSize);
    printf("Total: %d threads\n", gridSize * blockSize);
    printf(errors == 0 ? "PASS: all %d results correct\n"
                       : "FAIL: %d errors found\n",
           errors == 0 ? N : errors);

    // 9. Free all memory (GPU and CPU)
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return errors ? 1 : 0;
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o vec_add vec_add.cu && ./vec_add

## 2. What Every Line Does

| Line | What it is | Notes |
|------|-----------|-------|
| `__global__` | Kernel qualifier | Runs on GPU, callable from CPU |
| `cudaMalloc(&ptr, bytes)` | GPU memory allocation | Like `malloc` but on device (HBM) |
| `cudaMemcpy(..., H2D)` | Host → Device transfer | Over PCIe ~32 GB/s — minimize these |
| `<<<gridSize, blockSize>>>` | Launch configuration | Grid dimensions, then block dimensions |
| `blockIdx.x * blockDim.x + threadIdx.x` | Global thread ID | The fundamental indexing formula |
| `if (i < N)` | Boundary guard | Last block has (blockSize − N%blockSize) idle threads |
| `cudaDeviceSynchronize()` | CPU waits for GPU | Without this, D2H copy starts before kernel finishes |
| `cudaMemcpy(..., D2H)` | Device → Host transfer | Gets results back to CPU RAM |
| `cudaFree` | Free GPU memory | Must pair with every `cudaMalloc` |

### Why `blockSize = 256`?

The GPU executes threads in groups of **32 called warps** — the hardware's actual unit of execution.
Block size must be a multiple of 32. Common values:

| blockSize | Warps/block | Typical use |
|-----------|-------------|-------------|
| 32 | 1 | Very simple kernels |
| **256** | **8** | **Default starting point** |
| 512 | 16 | Higher occupancy potential |
| 1024 | 32 | Maximum — may limit register use |

You'll tune this in Notebook 02 using occupancy analysis.

In [ ]:
%%writefile vec_add_bench.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void vecAdd(float *A, float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) C[i] = A[i] + B[i];
}

int main() {
    const int    N     = 1 << 24;
    const size_t bytes = N * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes); cudaMalloc(&d_B, bytes); cudaMalloc(&d_C, bytes);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize  = (N + blockSize - 1) / blockSize;

    // Warmup: first launch has JIT-compilation overhead on the driver side
    vecAdd<<<gridSize, blockSize>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    // CUDA Events: GPU-side timer — no CPU-side synchronization overhead
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    const int REPS = 200;
    cudaEventRecord(start);
    for (int r = 0; r < REPS; r++)
        vecAdd<<<gridSize, blockSize>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);  // CPU blocks until stop event is recorded

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);  // total ms for all REPS
    ms /= REPS;                              // average ms per launch

    // Effective bandwidth: read A + read B + write C = 3 passes over N floats
    float bw = (3.0f * bytes) / (ms / 1000.0f) / 1e9f;

    printf("---------------------------------------------\n");
    printf("CUDA vecAdd  | N = 16,777,216 floats\n");
    printf("Time:        %.4f ms  (avg over %d runs)\n", ms, REPS);
    printf("Bandwidth:   %.1f GB/s\n", bw);
    printf("T4 peak:     ~300 GB/s\n");
    printf("Utilization: %.1f%%\n", bw / 300.0f * 100.0f);
    printf("---------------------------------------------\n");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B);
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o vec_add_bench vec_add_bench.cu && ./vec_add_bench

## 3. Benchmark vs. PyTorch

PyTorch's `A + B` on GPU compiles to a CUDA kernel essentially identical to ours.
Both are **memory-bandwidth bound** — the bottleneck is moving data through HBM, not arithmetic.

**Arithmetic intensity** of vector add:
- 1 FLOP (one addition) per 12 bytes read/written (two reads + one write × 4 bytes)
- T4 peak compute: ~8 TFLOPS FP32
- T4 peak memory BW: ~300 GB/s
- Compute ceiling: 8e12 / (1/12) = ... much higher than our kernel will ever need
- **Conclusion:** we are firmly memory-bound. Raw FLOPS don't matter here.

The number below should closely match your C kernel output above.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

N = 1 << 24  # 16,777,216 elements — same as the C kernel
A = torch.ones(N, device='cuda', dtype=torch.float32)
B = torch.full((N,), 2.0, device='cuda', dtype=torch.float32)

# Warmup
for _ in range(10):
    C = A + B
torch.cuda.synchronize()

# Benchmark with CUDA Events (same method as the C kernel)
start_evt = torch.cuda.Event(enable_timing=True)
end_evt   = torch.cuda.Event(enable_timing=True)

REPS = 200
start_evt.record()
for _ in range(REPS):
    C = A + B
end_evt.record()
torch.cuda.synchronize()

ms = start_evt.elapsed_time(end_evt) / REPS
bw = (3 * N * 4) / (ms / 1000) / 1e9  # 3 arrays × N elements × 4 bytes/float

print()
print("PyTorch vecAdd  | N = 16,777,216 floats")
print(f"Time:           {ms:.4f} ms  (avg over {REPS} runs)")
print(f"Bandwidth:      {bw:.1f} GB/s")
print(f"T4 peak:        ~300 GB/s")
print(f"Utilization:    {bw/300*100:.1f}%")
print()
print("Compare to the CUDA C number above.")
print("Both are close because vector add is purely memory-bound.")
print("Any difference is PyTorch framework overhead, not kernel quality.")


## Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| **Thread hierarchy** | Grid → Block → Thread; global index = `blockIdx.x * blockDim.x + threadIdx.x` |
| **Memory lifecycle** | `cudaMalloc` → `H2D memcpy` → kernel → `D2H memcpy` → `cudaFree` |
| **Launch config** | `<<<gridSize, blockSize>>>` — block size is a multiple of 32, usually 256 |
| **Boundary guard** | `if (i < N)` prevents out-of-bounds when N % blockSize ≠ 0 |
| **CUDA Events** | GPU-side timer — more accurate than `clock()` for short kernels |
| **Memory-bound kernel** | Vector add hits the HBM bandwidth wall — same result as PyTorch expected |

---

### What's next

**Notebook 02 (Matrix Multiply)** is where arithmetic intensity finally matters.

Naive GEMM re-reads the same matrix rows/columns from HBM thousands of times.
Tiled GEMM with shared memory caches those tiles on-chip — eliminating most HBM traffic
and achieving 10–50× improvement on compute-bound workloads.